# AI Sarthi — GovHealth-SLM QLoRA Fine-Tuning Notebook

This notebook trains a **GovHealth-SLM** LoRA adapter on `Qwen/Qwen2.5-1.5B-Instruct` or `Qwen/Qwen3-1.7B` using the verified 6,949 AI Sarthi public health instruction examples.

### Hardware requirements:
- Google Colab Free Tier with T4 GPU (15 GB VRAM)
- 4-bit NormalFloat4 (bitsandbytes) quantization
- Sequence length: 1,024 tokens

In [ ]:
# 1. Install dependencies
!pip install -q -U torch transformers peft bitsandbytes datasets trl accelerate

In [ ]:
# 2. Check GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 3. Upload or Download train.jsonl and val.jsonl
# If running in Colab, upload train.jsonl and val.jsonl from backend/training_data/
from google.colab import files
import os

if not os.path.exists("train.jsonl"):
    print("Please upload train.jsonl and val.jsonl:")
    files.upload()

In [ ]:
# 4. Run Training Script
!python train_qwen_qlora.py

In [ ]:
# 5. Test Inference with LoRA Adapter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(base_model, "./ai_sarthi_qwen_qlora")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

prompt = "<|im_start|>user\nWhat is the institutional birth rate in Pune according to NFHS-5?<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))